In [ ]:
%pip install tensorflow

In [142]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import tensorflow.keras as keras
import random

In [143]:
def build_q_network(state_shape, action_size):
    model = tf.keras.Sequential([
        layers.Input(shape=(state_shape, )),
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(action_size, activation=None) #Outputs Q-values for each action
    ])
    return model

In [144]:
class ReplayBuffer:
    def __init__(self):
        self.buffer = []
    def store(self, state, action, reward, next_state, done):
        if len(self.buffer) == 5000:
            self.buffer.pop(0)
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))

In [145]:
def traded_resources(identifier):
  collection = ['w', 'h', 'o', 't', 'b']
  if identifier < 4:
    r1 = 0
  elif identifier < 8:
    r1 = 1
    identifier = identifier - 4
  elif identifier < 12:
    r1 = 2
    identifier = identifier - 8
  elif identifier < 16:
    r1 = 3
    identifier = identifier - 12
  else:
    r1 = 4
    identifier = identifier - 16

  new_collection = collection.copy()
  new_collection.remove(collection[r1])
  res2 = new_collection[identifier]
  r2 = collection.index(res2)
  return [r1, r2] #r1 is the resource being demanded, r2 is the resource being given


def conduct_trade(a1, a2, proposal, flag):
  if flag:
    if a1.resources[proposal[2]] == 0:
      return False
    a1.resources[proposal[1]] = a1.resources[proposal[1]] + 1
    a2.resources[proposal[2]] = a2.resources[proposal[2]] + 1
    a1.resources[proposal[2]] = a1.resources[proposal[2]] - 1
    a2.resources[proposal[1]] = a2.resources[proposal[1]] - 1
    a1.loyalty[a2.id] = np.clip(a1.loyalty[a2.id] + 5, None, 100)
    a2.loyalty[a1.id] = np.clip(a2.loyalty[a1.id] + 3, None, 100)
    a1.last_traded = a2.id
    a2.last_traded = a1.id
    return True

In [146]:
class Ruler:
  def __init__(self, id):
    self.id = id
    self.harvest = np.ones(5) #Format: 'w', 'h', 'o', 't', 'b'
    self.upgrade = np.ones(5) #Format: 'w', 'h', 'o', 't', 'b'
    self.loyalty = np.ones(5)*50 #Loyalty relations with other rulers
    self.last_traded = None
    self.level = 1
    self.resources = np.zeros(5)
    self.rounds_played = 0
    self.last_action = None
    self.last_reward = None
    self.succ_trades = 0
    self.trade_attempts = 0
    self.avg_reward = 0

    self.epsilon = 1.0 #Exploration factor
    self.optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    self.memory = ReplayBuffer() #Stores memory of one whole game
    self.gamma = 0.9 #Discount rate
    self.model = build_q_network(9, 81)
    self.model1 = keras.models.clone_model(self.model)
    self.model1.build(self.model.input_shape)
    self.model1.set_weights(self.model.get_weights())

  def init_beatles(self):
    #Requirements are temporarily modified to make initial learning easier
    if self.id == 0: #John
      self.harvest = np.array([0, 0, 1, 1, 0])
      self.upgrade = np.array([0, 1, 0, 1, 1])
    elif self.id == 1: #Ringo
      self.harvest = np.array([1, 0, 0, 0, 1])
      self.upgrade = np.array([1, 0, 1, 0, 1])
    elif self.id == 2: #George
      self.harvest = np.array([1, 1, 0, 0, 0])
      self.upgrade = np.array([1, 1, 1, 0, 0])
    elif self.id == 3: #Paul
      self.harvest = np.array([0, 1, 0, 0, 1])
      self.upgrade = np.array([0, 0, 1, 1, 1])
    elif self.id == 4: #Pete
      self.harvest = np.array([1, 0, 1, 0, 0])
      self.upgrade = np.array([0, 1, 0, 1, 1])

  def get_legal_actions(self):
    legal_actions = [0]
    if self.resources[0] > 0:
      legal_actions.extend([5, 9, 13, 17]) #1, 5, 9, 13, 17
      legal_actions.extend([25, 29, 33, 37])
      legal_actions.extend([45, 49, 53, 57])
      legal_actions.extend([65, 69, 73, 77])
    if self.resources[1] > 0:
      legal_actions.extend([1, 10, 14, 18])
      legal_actions.extend([21, 30, 34, 38])
      legal_actions.extend([41, 50, 54, 58])
      legal_actions.extend([61, 70, 74, 78])
    if self.resources[2] > 0:
      legal_actions.extend([2, 6, 15, 19])
      legal_actions.extend([22, 26, 35, 39])
      legal_actions.extend([42, 46, 55, 59])
      legal_actions.extend([62, 66, 75, 79])
    if self.resources[3] > 0:
      legal_actions.extend([3, 7, 11, 20])
      legal_actions.extend([23, 27, 31, 40])
      legal_actions.extend([43, 47, 51, 60])
      legal_actions.extend([63, 67, 71, 80])
    if self.resources[4] > 0:
      legal_actions.extend([4, 8, 12, 16])
      legal_actions.extend([24, 28, 32, 36])
      legal_actions.extend([44, 48, 52, 56])
      legal_actions.extend([64, 68, 72, 76])

    return legal_actions

  def consider_proposal(self, proposal): #Harcoded rule for accepting or rejecting a proposed trade

    #Proposal --> [id of proposing agent, resource being demanded, resource being given]
    if self.resources[proposal[1]] < 1:
      return False

    if self.upgrade[proposal[2]] > 0: # and self.resources[proposal[2]] < self.resources[proposal[2]]:
      if self.harvest[proposal[1]] > 0:
        if self.harvest[proposal[2]] == 0:
          return True #Needs the given resource, does not harvest it, and harvests the proposed resource
        else:
          if np.random.random()*100 < self.loyalty[proposal[0]]:
            return True
          else:
            return False #Random selection
      else:
        if self.loyalty[proposal[0]] > 90 and self.upgrade[proposal[1]] == 0:
          return True #Feels obliged due to loyalty relations, and does not need the demanded resource
        else:
          return False

    else:
      return False

  def harvest_resources(self):
    self.resources = self.resources + self.harvest

  def upgrade_level(self):
    check = np.all(self.resources - self.upgrade >= 0)
    if check:
      self.resources = self.resources - self.upgrade
      self.level = self.level + 1

    return check


  def take_action(self, all_rulers):
    reward = 0
    state = np.zeros(9)
    state[0:5] = self.resources
    i = 5
    for j in range(5):
      if j == self.id:
        continue
      state[i] = self.loyalty[j]
      i = i + 1

    legal_actions = self.get_legal_actions()
    actions_np = (self.model(state.reshape(1, 9))).numpy()
    for action in range(1, 81):
      if action not in legal_actions:
        actions_np[0, action] = -np.inf #Illegal actions should get -inf Q-value
    actions = tf.convert_to_tensor(actions_np)

    if np.random.random() < self.epsilon:
      action = np.random.choice(legal_actions)
    else:
      action = np.argmax(actions)
    self.epsilon = self.epsilon*0.99
    self.epsilon = np.clip(self.epsilon, 0.1, 1.0)

    self.last_action = action
      
    if action == 0: #Pure harvesting
      self.harvest_resources()
    else: #Trading
      self.trade_attempts = self.trade_attempts + 1
      mult = (action-1)//20
      traded_resource = traded_resources((action-1)%20)
      proposal = [self.id, traded_resource[0], traded_resource[1]]
      og = self.resources[traded_resource[0]]
      flag = all_rulers[(self.id+mult+1)%5].consider_proposal(proposal)
      check = conduct_trade(self, all_rulers[(self.id+mult+1)%5], proposal, flag)
      new = self.resources[traded_resource[0]]
      if self.upgrade[traded_resource[0]] > 0:
        reward = reward + (new-og)*7
      if check:
        reward = reward + 2
        self.succ_trades = self.succ_trades + 1

    if self.upgrade_level():
      reward = reward + 20
    else:
      reward = reward + 0

    self.last_reward = reward
    self.avg_reward = (self.avg_reward*(self.rounds_played-1) + reward)/(self.rounds_played)
    
    next_state = np.zeros(9)
    next_state[0:5] = self.resources
    i = 5
    for j in range(5):
      if j == self.id:
        continue
      next_state[i] = self.loyalty[j]
      i = i + 1
    return (state, action, reward, next_state)

    
  def train(self):
    #Get samples
    samples = self.memory.sample(20)

    #Update target model
    if self.rounds_played % 10 == 0:
      self.model1.set_weights(self.model.get_weights())

    with tf.GradientTape() as tape:
      #Get the loss
      states, actions, rewards, next_states, done_check = samples
      no_samples = states.shape[0]
      loss = 0
      q_vals = []
      for i in range(no_samples):
        state = states[i]
        action = actions[i]
        reward = rewards[i]
        done = done_check[i]
        next_state = next_states[i]
        q_val_check = self.model(state.reshape(1, 9))[0, action]
        q_vals.append(q_val_check)
        if not done:
          best = tf.argmax(self.model(next_state.reshape(1,9)), axis=1)
          loss = loss + (self.model(state.reshape(1, 9))[0, action] - reward - self.gamma*(self.model1(next_state.reshape(1,9))[0, best[0]]))**2
        else:
          loss = loss + (self.model(state.reshape(1, 9))[0, action] - reward)**2 #No further states

      loss = loss/no_samples

    #Backprop
    grads = tape.gradient(loss, self.model.trainable_variables)
    self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
    return (loss, np.mean(q_vals), actions)

In [147]:
def run_round(all_rulers):
  for i in range(5):
    done = False
    if all_rulers[i].rounds_played == 499:
      done = True
    all_rulers[i].rounds_played = all_rulers[i].rounds_played + 1 #Increments round counter
    state, action, reward, next_state = all_rulers[i].take_action(all_rulers)
    all_rulers[i].memory.store(state, action, reward, next_state, done) #Stores in memory buffer

def jealousy_check(all_rulers):
  bf = []
  for ruler in all_rulers:
    bf.append(np.argmax(ruler.loyalty))
  for i in range(len(bf)):
    if bf[bf[i]] != i:
      all_rulers[i].loyalty[bf[i]] = all_rulers[i].loyalty[bf[i]]*0.7

def train_all(all_rulers):
  losses = []
  q_vals = []
  all_actions = []
  for ruler in all_rulers:
    loss = 0
    for _ in range(5):
      train_loss, q_val_mean, actions = ruler.train()
      loss = loss + train_loss
      q_vals.append(q_val_mean)
    loss = loss/5
    losses.append(loss)
    all_actions.append(actions)
  return losses, q_vals, all_actions #actions is of the last of the 5 runs


def bound_resources(all_rulers):
  for ruler in all_rulers:
    if np.sum(ruler.resources) >= 10:
      ruler.resources = -(ruler.resources//-2)

Now we run a whole game, consisting of 500 rounds.

In [148]:
def run_game(all_rulers):
  for ruler in all_rulers:
    ruler.resources = np.zeros(5)
    ruler.level = 1
    ruler.rounds_played = 0
    ruler.epsilon = 1.0
    ruler.succ_trades = 0
    ruler.trade_attempts = 0
    ruler.avg_reward = 0
    ruler.loyalty = np.ones(5)*50
  for i in range(200):
    bound_resources(all_rulers)
    run_round(all_rulers)
    #jealousy_check(all_rulers)
    if i > 19:
      losses, q_vals, all_actions = train_all(all_rulers)
      print(i)
      print(losses)
      print(all_rulers[0].avg_reward)
      print(all_rulers[1].avg_reward)
      print(all_rulers[2].avg_reward)
      print(all_rulers[3].avg_reward)
      print(all_rulers[4].avg_reward)
      print('\n')
      #print(q_vals)

Let's create certain characters, who get specific resources, but need more than just those to upgrade their kingdoms. This forces them to trade with the others to succeed. The Beatles

In [149]:
all_rulers = [Ruler(0), Ruler(1), Ruler(2), Ruler(3), Ruler(4)]
for player in all_rulers:
  player.init_beatles()

In [150]:
all_rulers[0].model.load_weights('ruler1_weights.weights.h5')
all_rulers[1].model.load_weights('ruler2_weights.weights.h5')
all_rulers[2].model.load_weights('ruler3_weights.weights.h5')
all_rulers[3].model.load_weights('ruler4_weights.weights.h5')
all_rulers[4].model.load_weights('ruler5_weights.weights.h5')

In [ ]:
run_game(all_rulers)

In [151]:
def print_levels(all_rulers):
  print("John's level: " + str(all_rulers[0].level))
  print("Ringo's level: " + str(all_rulers[1].level))
  print("George's level: " + str(all_rulers[2].level))
  print("Paul's level: " + str(all_rulers[3].level))
  print("Pete's level: " + str(all_rulers[4].level))

print_levels(all_rulers)

John's level: 1
Ringo's level: 1
George's level: 1
Paul's level: 1
Pete's level: 1


In [141]:
all_rulers[0].model.save_weights('ruler1_weights.weights.h5')
all_rulers[1].model.save_weights('ruler2_weights.weights.h5')
all_rulers[2].model.save_weights('ruler3_weights.weights.h5')
all_rulers[3].model.save_weights('ruler4_weights.weights.h5')
all_rulers[4].model.save_weights('ruler5_weights.weights.h5')

In [ ]:
def run_test_game(all_rulers):
    all_rulers[0].model.load_weights('ruler1_weights.weights.h5')
    all_rulers[1].model.load_weights('ruler2_weights.weights.h5')
    all_rulers[2].model.load_weights('ruler3_weights.weights.h5')
    all_rulers[3].model.load_weights('ruler4_weights.weights.h5')
    all_rulers[4].model.load_weights('ruler5_weights.weights.h5')
    for ruler in all_rulers:
        ruler.epsilon = 0
        ruler.resources = np.zeros(5)
        ruler.level = 1
        ruler.rounds_played = 0
        ruler.succ_trades = 0
        ruler.trade_attempts = 0
        ruler.avg_reward = 0
        
    for i in range(200):
        bound_resources(all_rulers)
        run_round(all_rulers)
        for ruler in all_rulers:
            print(ruler.last_action)
            print(ruler.last_reward)
            print('\n')

run_test_game(all_rulers)
for ruler in all_rulers:
    ruler.epsilon = 1.0

In [ ]:
def get_metrics(all_rulers):
    for ruler in all_rulers:
        print([ruler.id, ruler.avg_reward, (ruler.succ_trades/ruler.trade_attempts)])

get_metrics(all_rulers)

In [ ]:
for _ in range(50):
    run_game(all_rulers)
    get_metrics(all_rulers)

20
[<tf.Tensor: shape=(), dtype=float32, numpy=13048.669921875>, <tf.Tensor: shape=(), dtype=float32, numpy=18913.359375>, <tf.Tensor: shape=(), dtype=float32, numpy=37772.3828125>, <tf.Tensor: shape=(), dtype=float32, numpy=30474.568359375>, <tf.Tensor: shape=(), dtype=float32, numpy=8838.689453125>]
0.5238095238095238
0.9523809523809523
0.0
0.42857142857142855
0.42857142857142855


21
[<tf.Tensor: shape=(), dtype=float32, numpy=12234.9814453125>, <tf.Tensor: shape=(), dtype=float32, numpy=17957.34765625>, <tf.Tensor: shape=(), dtype=float32, numpy=34662.578125>, <tf.Tensor: shape=(), dtype=float32, numpy=26383.353515625>, <tf.Tensor: shape=(), dtype=float32, numpy=8365.615234375>]
0.5
0.9090909090909091
0.0
0.4090909090909091
0.4090909090909091


22
[<tf.Tensor: shape=(), dtype=float32, numpy=11536.33984375>, <tf.Tensor: shape=(), dtype=float32, numpy=17180.6640625>, <tf.Tensor: shape=(), dtype=float32, numpy=31070.20703125>, <tf.Tensor: shape=(), dtype=float32, numpy=24303.89453125>